### **TRABAJO PRÁCTICO N° 3: Redes Convolucionales, Detección de Objetos y Redes Recurrentes - Problema 3**
---
 1er cuatrimestre - Año 2026

| **Integrantes**           | **Legajo** |
|---------------------------|------------|
| Martinez Dufour, Caterina | M-7169/2   |
| Grimaldi, Damián Daniel   | G-5977/3   |
| Tapia, Fabrizio           | T-3095/3   |

**Docentes:** *Salvañá, Leandro* - *Fernández, Florencia*

### 1. Descripción del dataset
---

En este ejercicio deben construir una Red Neuronal Convolucional-Recurrente (CRNN) para reconocer comandos de voz a partir de grabaciones de audio de un segundo. El dataset contiene grabaciones pronunciadas por muchos hablantes distintos en condiciones acusticas reales (ruido de fondo, variaciones de micrófono), lo que hace que el modelado temporal de la RNN tenga valor real: comandos como down o stop son secuencias fon ́eticas con estructura temporal que una red sin memoria no puede capturar correctamente.

La arquitectura sigue un esquema many-to-one: la CNN procesa cada ventana temporal del espectrograma de forma independiente y produce un vector de características. la secuencia de esos vectores entra a la RNN, que modela cómo los patrones acústicos evolucionan en el tiempo. el  ́ultimo estado oculto se clasifica con una capa densa.

Reducción del problema: Si el entrenamiento completo resulta demasiado pesado para los recursos disponibles, pueden trabajar con un subconjunto de 4 a 6 clases (ej. yes, no, stop, go). Deben justificar la selección en el informe. Se recomienda guardar checkpoints regularmente para poder retomar el entrenamiento ante interrupciones.


#### Variables del conjunto de datos  
**Variables explicativas (features):** - Archivos de audio en formato WAV de 1 segundo de duración, con una frecuencia de muestreo de 16,000 Hz (16 kHz).
 
**Variable objetivo (target):** - Clasificación multiclase (10 clases independientes) para diferenciar entre los comandos de voz sugeridos: 'down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up' y 'yes'.

#### Librerias

In [1]:
!pip install librosa


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install gdown


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Librerías
import pandas as pd
import seaborn as sns
import optuna
import gdown
import zipfile
import os
import random, numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix

In [4]:
import numpy as np
import tensorflow as tf
import librosa 

In [5]:
import os
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from IPython import display

In [6]:
import shutil  # borra carpetas completas

Fijar una semilla aleatoria al inicio de cada experimento para que los resultados sean reproducibles. Esto incluye las semillas de Python, NumPy y el framework de deep learning utilizado:

In [7]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### 2. Carga del dataset
---

Para comenzar con el desarrollo, cargamos el dataset a través del enlace de Google Drive: "https://drive.google.com/file/d/1w5drimnOidgeVQu-HrdvrIeZq6u3QnM4/view". Descargamos desde el Drive el dataset de los audios en formato zip, lo descomprimimos y lo almacenamos en la carpeta "problema_3/dataset_problema3":

In [8]:
# ID de tu archivo de Drive
file_id = '1w5drimnOidgeVQu-HrdvrIeZq6u3QnM4'
url = f'https://drive.google.com/uc?id={file_id}'
archivo_descargado = 'dataset_audios.zip' 

# Descargar el archivo
gdown.download(url, archivo_descargado, quiet=False)
# Descomprimir el archivo en la carpeta
carpeta_destino = 'problema_3/dataset_problema3'

# Creamos la carpeta si no existe
if not os.path.exists(carpeta_destino):
    os.makedirs(carpeta_destino)

# Extraemos el contenido
try:
    with zipfile.ZipFile(archivo_descargado, 'r') as zip_ref:
        zip_ref.extractall(carpeta_destino)
    # Eliminar el archivo .zip descargado
    os.remove(archivo_descargado) 
    
except zipfile.BadZipFile:
    print("Error.")

Downloading...
From (original): https://drive.google.com/uc?id=1w5drimnOidgeVQu-HrdvrIeZq6u3QnM4
From (redirected): https://drive.google.com/uc?id=1w5drimnOidgeVQu-HrdvrIeZq6u3QnM4&confirm=t&uuid=b8c2ee8d-9a94-4299-ac15-85ed85a3ecb1
To: c:\Users\Usuario\cnn-rnn-objectdetection-deeplearning\problema_3\dataset_audios.zip
100%|██████████| 2.42G/2.42G [06:50<00:00, 5.90MB/s]


Clases sugeridas (10): yes, no, up, down, left, right, on, off, stop, go

In [16]:
DATA_PATH = 'problema_3/dataset_problema3' 
comandos_objetivo = ['down', 'go', 'left', 'no', 'right', 'stop', 'up', 'yes', 'on', 'off']
archivos_control = ['validation_list.txt', 'testing_list.txt']

for elemento in os.listdir(DATA_PATH):
    ruta_completa = os.path.join(DATA_PATH, elemento)
    
    # Borramos carpetas de palabras que no nos interesen
    if os.path.isdir(ruta_completa) and (elemento not in comandos_objetivo):
        shutil.rmtree(ruta_completa)
    # Borramos archivos de texto que no sean las listas de partición oficiales
    elif os.path.isfile(ruta_completa) and (elemento not in archivos_control):
        os.remove(ruta_completa)

In [20]:
commands = [d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))]
print("Carpetas de comandos sugeridos:", commands)

Carpetas de comandos sugeridos: ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes']


### 3. Preparación de datos
---

Dividimos el dataset en conjuntos de entrenamiento, validación y prueba utilizando una proporción de 80:10:10

In [21]:
test_list_oficial = []
val_list_oficial = []

with open(os.path.join(DATA_PATH, 'testing_list.txt'), 'r') as f:
    lineas_test = f.readlines()
    for line in lineas_test:
        test_list_oficial.append(line)

with open(os.path.join(DATA_PATH, 'validation_list.txt'), 'r') as f:
    lineas_val = f.readlines()
    for line in lineas_val:
        val_list_oficial.append(line)

train_files = []
val_files = []
test_files = []

# Recorremos las carpetas para armar los conjuntos definitivos
for label in commands:
    label_dir = os.path.join(DATA_PATH, label)
    for fname in os.listdir(label_dir):
        # Creamos la ruta relativa con '\n' para que coincida con el .txt
        rel_path_con_salto = f"{label}/{fname}\n"
        full_path = os.path.join(label_dir, fname)
        # Evaluamos la pertenencia usando la cadena con el salto de línea incorporado
        if rel_path_con_salto in test_list_oficial:
            test_files.append(full_path)
        elif rel_path_con_salto in val_list_oficial:
            val_files.append(full_path)
        else:
            train_files.append(full_path)

print('Training set size:', len(train_files))
print('Validation set size:', len(val_files))
print('Test set size:', len(test_files))

Training set size: 30769
Validation set size: 3703
Test set size: 4074


### 4. pipeline CNN + LSTM (continuar)
---

In [ ]:
# Pipeline completo: mel-espectrograma con librosa y clasificador CNN + LSTM.
# entrada: Audio - encoder típico: CNN 1D / RNN  - salida: Transcripción

# --- Preprocesamiento: audio crudo -> mel-espectrograma ---
def audio_to_melspectrogram(
    path: str,
    sr: int = 16000,       # sample rate
    n_mels: int = 64,      # bins de frecuencia (mel)
    n_fft: int = 512,      # longitud de la ventana FFT
    hop_length: int = 160  # desplazamiento (10 ms a 16 kHz)
) -> np.ndarray:
    """Retorna mel-espectrograma de forma (T, n_mels)."""
    y, _ = librosa.load(path, sr=sr)
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels,
        n_fft=n_fft, hop_length=hop_length
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    return S_db.T  # transponer a (T, n_mels)

# --- Arquitectura CNN + LSTM ---
def build_audio_classifier(
    T: int,         # frames temporales
    n_mels: int,    # bins de frecuencia
    K: int,         # numero de clases
    lstm_units: int = 128
):
    inputs = tf.keras.Input(shape=(T, n_mels, 1))

    # Etapa 1: CNN sobre el eje frecuencial en cada frame
    x = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Conv1D(32, kernel_size=3,
                               activation="relu",
                               padding="same")
    )(inputs)
    x = tf.keras.layers.TimeDistributed(
        tf.keras.layers.GlobalAveragePooling1D()
    )(x)
    # Salida: (batch, T, 32)

    # Etapa 2: modelado temporal
    x = tf.keras.layers.LSTM(lstm_units)(x)

    # Etapa 3: clasificacion
    outputs = tf.keras.layers.Dense(
        K, activation="softmax"
    )(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Construcción y verificación del modelo recurrente/convolucional
model = build_audio_classifier(T=100, n_mels=64, K=10)
model.summary()